In [15]:
import pprint
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

In [16]:
from pydantic import BaseModel
from typing import List, Optional

# 1. We define what our "Tray" (State) is allowed to hold.
class ComplianceState(BaseModel):
    
    # This holds the raw text file the user uploaded. 
    # It must be a string (text).
    raw_document: str
    
    # This starts as an empty list []. 
    # Agent 1 (Retriever) will eventually fill this with relevant laws.
    retrieved_policies: List[str] = []
    
    # This starts as an empty list []. 
    # Agent 2 (Evaluator) will fill this with dictionaries of violations it finds.
    compliance_violations: List[dict] = []
    
    # This starts as None (empty). 
    # Agent 3 (Writer) will fill this with the final text report at the very end.
    final_report: Optional[str] = None
    
    # This tracks where the tray is currently sitting in the shop.
    # We initialize it to start at the "Retrieval" step.
    current_step: str = "Retrieval"

In [17]:
def process_pdf_to_documents(file_path: str):
    """
    Loads a PDF and returns a list of LangChain Document objects.
    Each Document contains the page_content and metadata (page number).
    """
    # 1. Initialize the loader
    loader = PyPDFLoader(file_path)
    
    # 2. Load the data
    # By default, mode="page" will return a list where each item 
    # is a Document representing one page of the PDF.
    documents = loader.load()
    
    return documents

In [18]:
processed_documents = process_pdf_to_documents("./src/GDPR.pdf")

In [19]:
(processed_documents[0].page_content)

'REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL\nof 13 June 2024\nlaying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, \n(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and \nDirectiv es 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Ar tif icial Intelligence A ct)\n(T ext with EEA relevance)\nTHE EUR OPEAN P ARLIAMENT AND THE COUNCIL OF THE EUR OPEAN UNION,\nHaving regard to the T reaty on the Functioning of the European Union, and in par ticular Ar ticles 16 and 114 thereof,\nHaving regard to the proposal from the European Commission,\nAf ter transmission of the draf t legislative act to the national parliaments,\nHaving regard to the opinion of the European Economic and Social Committe e (\n1\n),\nHaving regard to the opinion of the European Central Bank (\n2\n),\nHaving regard to the opinion of the Committee of the Regions (\n3\n),\nA cting in accordance with the ord

In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [21]:
chunks = text_splitter.split_documents(processed_documents)
chunks[0].page_content

'REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL\nof 13 June 2024\nlaying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, \n(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and \nDirectiv es 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Ar tif icial Intelligence A ct)\n(T ext with EEA relevance)\nTHE EUR OPEAN P ARLIAMENT AND THE COUNCIL OF THE EUR OPEAN UNION,'

In [22]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    collection_name="gdpr_collection",
    embedding=OllamaEmbeddings(model="nomic-embed-text"),
    persist_directory="./chroma_langchain_db",
)

print("Database seeded successfully!")

Database seeded successfully!


In [11]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel


In [ ]:
def retrieve_relevant_policies(state: ComplianceState):
    """
    Retrieves relevant policies from the vector store based on the query.
    """
    vector_db = Chroma(
        persist_directory="./chroma_langchain_db",
        collection_name="gdpr_collection")
    
    relevant_docs = vector_db.similarity_search(state.raw_document, k=5)

    state.retrieved_policies = [doc.page_content for doc in relevant_docs]
    state.current_step = "Evaluation"

    return state


In [ ]:
app = FastAPI(title="Corporate Compliance Agent API", version="1.0")